# **Домашнее задание 1.** Векторные представления текстов [10 баллов]

## **Часть 1.** Классификация 16S рРНК [5 баллов]

Представим, что перед нами стоит задача определения видового состава микроорганизмов в некотором образце. Один из наиболее простых и распространенных подходов к ее решению — секвенирование 16S рРНК (точнее, кодирующего ее гена) и последующее определение таксономической принадлежности полученных последовательностей. Для проведения такой классификации было разработано большое количество различных методов. В этом домашнем задании мы реализуем упрощенную версию одного из них — **RDP Classifier** ([статья](https://doi.org/10.1128/AEM.00062-07), [GitHub](https://github.com/rdpstaff/classifier), [SourceForge](https://sourceforge.net/projects/rdp-classifier/)).

В основе RDP Classifier лежит наивный байесовский классификатор, использующий в качестве признаков **k-меры** длины 8. Чтобы лучше понять, как он работает, заметим, что последовательности нуклеотидов в некотором смысле тоже являются текстами. Тогда k-меры оказываются не более чем символьными **n-граммами**, а значит, поверх них мы можем построить уже знакомое нам **BoW**-представление. Остальное — лишь дело техники. Давайте приступать.


### 1.1. Токенизация [0.5 балла]

Начнем с токенизатора. Он должен разбивать последовательность на набор перекрывающихся k-меров с шагом 1. Например, при $k = 3$ последовательность `ACGTA` превратится в `ACG`, `CGT`, `GTA`.

In [ ]:
class KMerTokenizer:
    def __init__(self, k: int) -> None:
        self.k = k

    def __call__(self, sequence: str) -> list[str]:
        kmers: list[str] = []

        ...  # YOUR CODE HERE

        return kmers

In [ ]:
_tokenizer = KMerTokenizer(2)
assert _tokenizer("ACGT") == ["AC", "CG", "GT"]

_tokenizer = KMerTokenizer(3)
assert _tokenizer("ACGTA") == ["ACG", "CGT", "GTA"]

_tokenizer = KMerTokenizer(4)
assert _tokenizer("ACGTAC") == ["ACGT", "CGTA", "GTAC"]

_tokenizer = KMerTokenizer(3)
assert _tokenizer("ACG") == ["ACG"]
assert _tokenizer("AC") == []

### 1.2. Bag-of-Words [2 балла]

Теперь приступим к реализации Bag-of-Words. Но для начала подумаем вот о чем: сколько уникальных k-меров длины 8 существует? Если предположить, что в алфавите только 5 символов (4 основных нуклеотида и символ N, обозначающий неопределенный нуклеотид), получим $5^8 = 390\,625$. В худшем случае ровно столько значений будет содержать одна строка нашей матрицы BoW. При использовании 32 бит (4 байт) на каждое целое число на ее хранение потребуется $390\,625 \cdot 4 = 1\,562\,500$ байт, то есть примерно **1,6 MB**! И это только на одну строку! Таким образом, вся матрица может занимать десятки гигабайт оперативной памяти.

Разумеется, в реальности все чуть менее драматично. Например, многие k-меры могут вообще не встречаться в нашем датасете. Тем не менее, проблема хранения столь большой матрицы остается актуальной.

Примечательно, что каждая отдельная последовательность содержит лишь малую часть всех возможных k-меров, а значит, большинство значений в итоговой матрице будут нулевыми. Чтобы не расходовать на них память, лучше использовать разреженный (англ. *sparse*) формат хранения данных, при котором хранятся лишь ненулевые значения и их индексы в матрице.

**Примечание:** мы будем использовать модуль `scipy.sparse` из библиотеки SciPy. Ранее активно применялись разреженные матрицы (англ. *sparse matrices*), но сейчас SciPy переходит на разреженные массивы (англ. *sparse arrays*), а старый интерфейс планируется объявить устаревшим (deprecated). Поэтому и мы воспользуемся классом `csr_array`.

**Полезные ссылки:**

- [Документация Compressed Sparse Row array (`csr_array`)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_array.html)
- [Документация Compressed Sparse Row matrix (`csr_matrix`)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)
- [Миграция с sparse matrices на sparse arrays](https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html)

Один из способов создать `csr_array` — передать значения и соответствующие им индексы строк и столбцов:

In [ ]:
import numpy as np
from scipy.sparse import csr_array

values = [1, 2, 3, 4]

rows = [0, 0, 1, 1]
cols = [0, 1, 2, 2]

sparse_array = csr_array((values, (rows, cols)), shape=(2, 3), dtype=np.int32)
type(sparse_array)

In [ ]:
sparse_array

In [ ]:
sparse_array.toarray()

Обратите внимание, что SciPy автоматически просуммировал значения на одинаковых позициях.

In [ ]:
from collections.abc import Callable
from typing import Self


class BagOfWordsVectorizer:
    def __init__(self, tokenizer: Callable[[str], list[str]]) -> None:
        self.tokenizer = tokenizer

    def fit(self, documents: list[str]) -> Self:
        vocabulary: dict[str, int] = {}

        for document in documents:
            for token in self.tokenizer(document):
                ...  # YOUR CODE HERE

        self.vocabulary = vocabulary
        return self

    def transform(self, documents: list[str]) -> csr_array:
        rows: list[int] = []
        cols: list[int] = []

        for i, document in enumerate(documents):
            for token in self.tokenizer(document):
                ...  # YOUR CODE HERE

        values = ...  # YOUR CODE HERE

        return csr_array((values, (rows, cols)), shape=(len(documents), len(self.vocabulary)), dtype=np.int32)

    def fit_transform(self, documents: list[str]) -> csr_array:
        return self.fit(documents).transform(documents)

In [ ]:
_vectorizer = BagOfWordsVectorizer(KMerTokenizer(2))
_X = _vectorizer.fit_transform(["ACGT", "ACAC"])

assert _vectorizer.vocabulary == {"AC": 0, "CG": 1, "GT": 2, "CA": 3}
assert np.array_equal(
    _X.toarray(),
    np.array(
        [
            [1, 1, 1, 0],
            [2, 0, 0, 1],
        ]
    ),
)
assert np.array_equal(_vectorizer.transform(["ACGA"]).toarray(), np.array([[1, 1, 0, 0]]))
assert _vectorizer.transform(["", "NNNN"]).shape == (2, 4)
assert _vectorizer.transform(["", "NNNN"]).nnz == 0

### 1.3. Наивный байесовский классификатор [2 балла]

Наконец, мы готовы перейти к реализации **Naive Bayes classifier**. Вспомним, что предсказание модели для документа $x$ имеет вид

$$
\hat y
=
\arg\max_y
\left[
\log P(y)
+
\sum_{j=1}^{|V|} x_j \log P(v_j \mid y)
\right],
$$

где $x_j$ — количество вхождений токена $v_j \in V$ в данный документ, а $P(v_j \mid y)$ — вероятность этого токена при условии класса $y$.

Остается учесть еще одну небольшую деталь. Если некоторый токен ни разу не встретился в обучающих данных для класса $y$, то оценка его вероятности окажется равна нулю, и при вычислении $\log P(v_j \mid y)$ вселенная схлопнется... а точнее, мы получим `np.float64(-inf)`. Чтобы избежать этого, добавим к числу вхождений каждого токена **псевдосчетчик** (англ. *pseudocount*) $\alpha > 0$. В общем случае такое сглаживание называется **аддитивным сглаживанием** (additive smoothing), или **сглаживанием Лидстоуна** (англ. *Lidstone smoothing*), а при $\alpha = 1$ — **сглаживанием Лапласа** (англ. *Laplace smoothing*).

In [ ]:
class NaiveBayesClassifier:
    """
    Условные обозначения:
        C — количество классов;
        V — размер словаря.
    """

    def __init__(self, alpha: float = 1.0) -> None:
        self.alpha = alpha

    def fit(self, X: csr_array, y: np.ndarray) -> Self:
        # HINT: используйте `np.unique`
        classes, class_counts = ...  # YOUR CODE HERE
        self.classes = classes

        # log P(y), вектор размерности (C,)
        self.log_py = ...  # YOUR CODE HERE

        feature_counts = np.zeros((len(classes), X.shape[1]), dtype=np.float32)

        for class_idx, class_ in enumerate(classes):
            feature_counts[class_idx] = X[y == class_].sum(axis=0)

        feature_counts += self.alpha  # аддитивное сглаживание

        # log P(v|y), матрица размерности (C, V)
        self.log_pvy = ...  # YOUR CODE HERE

        return self

    def predict(self, X: csr_array) -> np.ndarray:
        # HINT: используйте матричное умножение для эффективного вычисления
        scores = ...  # YOUR CODE HERE
        return self.classes[scores.argmax(axis=1)]

In [ ]:
_X = csr_array(
    [
        [2, 0, 0],
        [1, 1, 0],
        [0, 1, 2],
        [0, 0, 2],
    ],
    dtype=np.int32,
)
_y = np.array([-1, -1, 1, 1])

_classifier = NaiveBayesClassifier(alpha=1.0)
_classifier.fit(_X, _y)

assert np.array_equal(_classifier.classes, [-1, 1])
assert np.allclose(_classifier.log_py, np.log([0.5, 0.5]))
assert np.allclose(
    np.exp(_classifier.log_pvy),
    [
        [4 / 7, 2 / 7, 1 / 7],
        [1 / 8, 2 / 8, 5 / 8],
    ],
)

_preds = _classifier.predict(
    csr_array(
        [
            [2, 0, 0],
            [0, 0, 2],
        ],
        dtype=np.int32,
    )
)
assert np.array_equal(_preds, [-1, 1])

### 1.4. Общий сбор [0.5 балла]

Мы почти у цели. Осталось только обучить нашу модель на данных, которые использует настоящий RDP Classifier.

In [ ]:
# %%bash
# mkdir -p data
# wget https://zenodo.org/records/10367203/files/RDPClassifier_16S_trainsetNo19_QiimeFormat.zip -O data/RDPClassifier_16S_trainsetNo19_QiimeFormat.zip
# unzip -qo data/RDPClassifier_16S_trainsetNo19_QiimeFormat.zip -d data

Прочитаем идентификаторы и последовательности из FASTA.

In [ ]:
import pandas as pd
from Bio import SeqIO

records = {"id": [], "sequence": []}

for record in SeqIO.parse("data/RDPClassifier_16S_trainsetNo19_QiimeFormat/RefOTUs.fa", "fasta"):
    records["id"].append(record.id)
    records["sequence"].append(str(record.seq).upper())

records = pd.DataFrame(records)
records.head()

Теперь прочитаем таксономию и выделим название рода из поля `g__...`.

In [ ]:
taxonomy = pd.read_csv(
    "data/RDPClassifier_16S_trainsetNo19_QiimeFormat/Ref_taxonomy.txt",
    sep="\t",
    header=None,
    names=["id", "taxonomy"],
)
taxonomy["genus"] = taxonomy["taxonomy"].str.extract(r"g__([^;]+)", expand=False)
taxonomy.head()

In [ ]:
taxonomy["genus"].value_counts()

Немного упростим задачу и оставим только роды, для которых есть не менее 50 записей.

In [ ]:
taxonomy = taxonomy[taxonomy.groupby("genus").transform("size") >= 50]

Соединим последовательности с таксономией по идентификатору и удалим повторы одной и той же последовательности внутри рода.

In [ ]:
data = records.merge(taxonomy, on="id", how="inner")
data = data.drop_duplicates(["sequence", "genus"])
data.head()

In [ ]:
len(data)

Разобьем данные на обучающую и тестовую подвыборки.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, stratify=data["genus"], random_state=42)
len(train), len(test)

Построим Bag-of-Words на k-мерах длины 8.

In [ ]:
vectorizer = ...  # YOUR CODE HERE

X_train = ...  # YOUR CODE HERE
X_test = ...  # YOUR CODE HERE

X_train.shape, X_test.shape

In [ ]:
len(vectorizer.vocabulary)

In [ ]:
assert len(vectorizer.vocabulary) == 89166

assert X_train.shape == (6933, 89166)
assert X_test.shape == (1734, 89166)

Обучим классификатор.

In [ ]:
classifier = ...  # YOUR CODE HERE
...  # YOUR CODE HERE

Сделаем предсказания и посчитаем Precision, Recall и F1 для каждого рода.

In [ ]:
from sklearn.metrics import classification_report

preds = ...  # YOUR CODE HERE
print(classification_report(test["genus"], preds, zero_division=0))

Предположите, чем могут быть обусловлены столь высокие значения метрик.

## **Часть 2.** Поиск по статьям PubMed [5 баллов]

С классификацией разобрались. Теперь давайте подумаем, для каких еще задач могут пригодиться векторные представления текстов.

Рассмотрим в качестве примера базу данных PubMed. Все доступные публикации в ней мы будем называть **документами** (англ. documents). Пусть теперь у нас есть некоторый **запрос** (англ. query), например, «влияние картинок с котиками на эмоциональное состояние студентов перед сессией». Мы хотим найти документы, наиболее релевантные данному запросу. Такая задача называется **информационным поиском** (англ. Information Retrieval, IR) и является одним из ключевых приложений машинного обучения в современном мире.

В простейшем случае мы можем использовать для поиска векторные представления запроса и документов: посчитать сходство вектора запроса с векторами всех документов в базе и выбрать документы с наибольшим сходством.

<div style="color: seagreen;">
<p>
<strong>[Опционально]</strong> Приведем еще один пример IR. При работе с большими языковыми моделями (англ. <em>Large Language Models</em>, LLM) мы надеемся, что они обладают всей необходимой для выполнения задачи информацией. В противном случае модель может галлюцинировать (англ. <em>hallucination</em>), т.е. генерировать недостоверную информацию. Но правда состоит в том, что <strong>без дополнительного контекста LLM опираются лишь на информацию, усвоенную при обучении и сохраненную в параметрах модели</strong>.
</p>

<p>
А если мы, например, хотим, чтобы LLM-ассистент в банке отвечал на вопросы пользователей с учетом нормативных документов банка? Обучать на них модель и обновлять ее при каждом изменении было бы долго и дорого. Вместо этого можно собрать все необходимые документы в базу знаний, извлекать из нее по запросу пользователя самые релевантные и передавать их в качестве контекста LLM. Такой подход называется <strong>RAG</strong> (англ. <em>retrieval-augmented generation</em>).
</p>
</div>

### 2.1. Данные [0 баллов]

Файл `documents.txt` содержит аннотации 20 000 реальных статей из базы данных PubMed по 20 различным тематикам.

In [ ]:
with open("data/documents.txt", encoding="utf-8") as file:
    documents = file.read().splitlines()

len(documents)

In [ ]:
import textwrap

print(textwrap.fill(documents[0], width=80))

Файл `queries.csv` содержит 200 уникальных запросов и индексы наиболее релевантных им документов из `documents.txt`.

In [ ]:
queries = pd.read_csv("data/queries.csv")
queries.head()

In [ ]:
print("Query: {}".format(queries["query"].iloc[42]))
print(
    textwrap.fill(
        "Target document: {}".format(documents[queries["target"].iloc[42]]),
        width=80,
    )
)

### 2.2. Токенизация. Bag-of-Words [1 балл]

Для начала напишем токенизатор, который будет приводить текст к нижнему регистру, разбивать его на токены через `WordPunctTokenizer` из NLTK и удалять токены, состоящие только из пунктуации.

In [ ]:
from string import punctuation

from nltk import WordPunctTokenizer

tokenizer = WordPunctTokenizer()


def tokenize(text: str) -> list[str]:
    ...  # YOUR CODE HERE

In [ ]:
assert tokenize("Gene-expression\nin IL-18 cells.") == ["gene", "expression", "in", "il", "18", "cells"]
assert tokenize("...?!") == []
assert tokenize("") == []

In [ ]:
bow = ...  # YOUR CODE HERE

bow_documents = ...  # YOUR CODE HERE
bow_queries = ...  # YOUR CODE HERE

bow_documents.shape, bow_queries.shape

### 2.3. Поиск ближайших статей [1 балл]

Представления запросов и статей готовы. Теперь для каждого запроса мы можем посчитать его близость ко всем статьям.

Пусть $q$ — вектор запроса, а $d$ — вектор документа. Самый простой способ оценить их близость — посчитать скалярное произведение $q^\top d$. Он удобен еще и тем, что для всех документов эту операцию можно выполнить одним умножением вектора на матрицу, а при наличии нескольких запросов — матричным умножением.

Для каждого запроса найдите топ-5 ближайших к нему документов по скалярному произведению.

In [ ]:
def top_k(scores: np.ndarray, k: int) -> np.ndarray:
    if scores.ndim != 2:
        raise ValueError("`scores` must be 2-dimensional array.")

    ...  # YOUR CODE HERE

In [ ]:
_scores = np.array(
    [
        [0.2, 0.5, -1.0, 0.9],
        [-2.0, 0.0, 3.0, 7.0],
    ]
)

assert np.array_equal(top_k(_scores, k=2), [[3, 1], [3, 2]])
assert np.array_equal(top_k(_scores, k=1), [[3], [3]])
assert np.array_equal(top_k(_scores, k=4), [[3, 1, 0, 2], [3, 2, 1, 0]])

In [ ]:
top_5_bow = ...  # YOUR CODE HERE

In [ ]:
assert top_5_bow.shape == (200, 5)

### 2.4. Оценка качества [0 баллов]

Как оценить, насколько хорошо работает наш поиск? В области IR существует множество разнообразных метрик. Мы разберем самую простую из них — **HitRate@K**. @K означает, что мы рассматриваем только топ-K предсказаний.

HitRate@K показывает долю запросов, для которых правильный ответ есть среди топ-K предсказаний:

$$
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}\left[t_i \in \operatorname{top}_K(q_i)\right],
$$

где $N$ — число запросов, $q_i$ — запрос, $t_i$ — его целевой документ, а $\operatorname{top}_K(q_i)$ — множество топ-K документов для этого запроса. Индикатор $\mathbf{1}[\cdot]$ равен 1, если условие выполнено, и 0 иначе.

Рассмотрим пример:

| Запрос | Целевой документ | Предсказанные топ-3 документа |
| ------ | ---------------- | --------------------------- |
| 1      | A                | **A**, C, E                 |
| 2      | B                | F, **B**, D                 |
| 3      | C                | B, G, **C**                 |
| 4      | D                | A, E, F                     |

Документы в последнем столбце упорядочены по убыванию оценок близости. Тогда:

- HitRate@1 = $1/4 = 0{,}25$;
- HitRate@2 = $2/4 = 0{,}5$;
- HitRate@3 = $3/4 = 0{,}75$.

In [ ]:
from collections.abc import Sequence


def hit_rate(preds: np.ndarray, targets: np.ndarray, ks: int | Sequence[int]) -> dict[str, float]:
    if preds.ndim != 2:
        raise ValueError("`preds` must be a 2-dimensional array.")

    if targets.ndim != 1:
        raise ValueError("`targets` must be a 1-dimensional array.")

    if preds.shape[0] != targets.shape[0]:
        raise ValueError("`preds` and `targets` must contain the same number of queries.")

    ks = (ks,) if isinstance(ks, int) else tuple(ks)

    for k in ks:
        if k <= 0:
            raise ValueError("all values of `k` must be positive.")

        if k > preds.shape[1]:
            raise ValueError("`k` cannot exceed the number of predictions.")

    values = np.maximum.accumulate(preds == targets[:, None], axis=1).mean(axis=0)

    return {f"HitRate@{k}": float(values[k - 1]) for k in ks}

In [ ]:
hit_rate(top_5_bow, queries["target"].to_numpy(), ks=(1, 5))

### 2.5. Косинусная близость [1 балл]

Скалярное произведение имеет существенный недостаток — оно учитывает норму (длину) вектора. Таким образом, документ с большой нормой может оказаться ближе к запросу, даже если он менее релевантен.

Если учесть особенность построения BoW (мы добавляем единицу за каждый токен в тексте), получится, что более длинные тексты, как правило, имеют большую норму. Таким образом, на верхних позициях в выдаче могут оказываться просто более длинные документы.

Поэтому на практике вместо скалярного произведения часто используют **косинусную близость** (англ. *cosine similarity*):

$$
\operatorname{cos}(q, d)
=
\frac{q^\top d}{\|q\| \|d\|}
=
\left(\frac{q}{\|q\|}\right)^\top
\left(\frac{d}{\|d\|}\right).
$$

Заметим также, что, если исходные векторы нормированы заранее, их скалярное произведение равно косинусной близости. Для каждого запроса найдите топ-5 ближайших к нему документов по косинусной близости.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

top_5_bow_cos = ...  # YOUR CODE HERE
hit_rate(top_5_bow_cos, queries["target"].to_numpy(), ks=[1, 5])

### 2.6. TF-IDF [2 балла]

Мы помним, что BoW учитывает только частоту встречаемости каждого отдельного токена в тексте, но не его распространенность в корпусе. Таким образом, «шумные» слова, которые встречаются во многих статьях, могут оказывать значительное влияние на расчет близости векторов. Попробуем улучшить наш алгоритм поиска при помощи TF-IDF.

Пусть $N$ — число документов, а $\operatorname{TF}_{ij}$ — число вхождений токена $j$ в документ $i$. Определим $\operatorname{DF}_j$ — число документов, в которых встретился токен $j$:

$$
\operatorname{DF}_j = \sum_{i=1}^{N} \mathbf{1}[\operatorname{TF}_{ij} > 0].
$$

Обратите внимание, что повторения токена внутри одного документа $\operatorname{DF}_j$ не увеличивают. Теперь мы можем рассчитать $\operatorname{IDF}_j$:

$$
\operatorname{IDF}_j = \log\frac{N+1}{\operatorname{DF}_j+1}+1.
$$

Чем реже токен встречается в корпусе, тем больше его $\operatorname{IDF}_j$. Наконец, получим $\operatorname{TF-IDF}_{ij}$, перемножив два показателя:

$$
\operatorname{TF-IDF}_{ij} = \operatorname{TF}_{ij}\operatorname{IDF}_j.
$$

In [ ]:
class TfIdfTransformer:
    def fit(self, X: csr_array) -> Self:
        df = ...  # YOUR CODE HERE
        self.idf = ...  # YOUR CODE HERE
        return self

    def transform(self, X: csr_array) -> csr_array:
        ...  # YOUR CODE HERE

    def fit_transform(self, X: csr_array) -> csr_array:
        return self.fit(X).transform(X)

In [ ]:
_tfidf = TfIdfTransformer()

_X_train = csr_array(
    [
        [2, 1, 0, 0],
        [1, 0, 3, 0],
        [1, 2, 0, 0],
    ],
    dtype=np.int32,
)
_X_test = csr_array(
    [
        [0, 2, 0, 1],
        [0, 0, 0, 0],
    ]
)

_X_train_tfidf = _tfidf.fit_transform(_X_train)
_X_test_tfidf = _tfidf.transform(_X_test)

_expected_idf = np.array(
    [
        1,
        np.log(4 / 3) + 1,
        np.log(2) + 1,
        np.log(4) + 1,
    ]
)

assert np.allclose(_tfidf.idf, _expected_idf)
assert isinstance(_X_train_tfidf, csr_array)
assert np.allclose(
    _X_train_tfidf.toarray(),
    [
        [2, _expected_idf[1], 0, 0],
        [1, 0, 3 * _expected_idf[2], 0],
        [1, 2 * _expected_idf[1], 0, 0],
    ],
)
assert np.allclose(
    _X_test_tfidf.toarray(),
    [
        [0, 2 * _expected_idf[1], 0, _expected_idf[3]],
        [0, 0, 0, 0],
    ],
)

In [ ]:
tfidf = ...  # YOUR CODE HERE

tfidf_documents = ...  # YOUR CODE HERE
tfidf_queries = ...  # YOUR CODE HERE

In [ ]:
top_5_tfidf_cos = ...  # YOUR CODE HERE
hit_rate(top_5_tfidf_cos, queries["target"].to_numpy(), ks=(1, 5))

Сделайте вывод о том, какие методы позволили улучшить качество поиска.